<a href="https://colab.research.google.com/github/narakrm/northstar_databases_analytics/blob/main/Section5_R_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 5 — R Analytics
**NorthStar Urban Mobility and Logistics — Databases and Analytics Assignment**

This notebook applies statistical analysis and data visualisation in R using `dplyr` and `ggplot2`.

**Analyses covered:**
1. Descriptive statistics on key numeric fields
2. Correlation analysis — driver rating vs failure rate
3. Correlation analysis — loyalty score vs complaint frequency
4. Visualisation 1 — Delivery status breakdown by zone (stacked bar)
5. Visualisation 2 — Customer rating by hub (boxplot)
6. Visualisation 3 — Order value vs fuel cost by service type (scatter)
7. Visualisation 4 — Monthly order volume and failure rate (dual-axis line)
8. Visualisation 5 — Delivery duration by status (violin plot)

---
> Upload all NorthStar CSV files to `/content/` before running, or set `data_path` to your Google Drive path.

## Cell 1 — Install packages and load libraries

In [ ]:
install.packages(c("dplyr", "ggplot2", "lubridate", "tidyr", "scales", "knitr"), quiet = TRUE)

library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(scales)
library(knitr)

# Consistent ggplot2 theme for all charts
theme_northstar <- theme_minimal(base_size = 13) +
  theme(
    plot.title    = element_text(face = "bold", size = 14, hjust = 0),
    plot.subtitle = element_text(size = 11, color = "grey40", hjust = 0),
    plot.caption  = element_text(size = 9,  color = "grey50", hjust = 1),
    axis.title    = element_text(size = 11),
    panel.grid.minor = element_blank(),
    legend.position  = "bottom"
  )

northstar_colours <- c(
  "OnTime"  = "#2E7D32",
  "Delayed" = "#F9A825",
  "Failed"  = "#C62828"
)

cat("Libraries loaded successfully\n")

## Cell 2 — Load and clean data

In [ ]:
data_path <- "/content/"

orders     <- read.csv(paste0(data_path, "orders.csv"),      stringsAsFactors = FALSE)
deliveries <- read.csv(paste0(data_path, "deliveries.csv"),  stringsAsFactors = FALSE)
drivers    <- read.csv(paste0(data_path, "drivers.csv"),     stringsAsFactors = FALSE)
customers  <- read.csv(paste0(data_path, "customers.csv"),   stringsAsFactors = FALSE)
complaints <- read.csv(paste0(data_path, "complaints.csv"),  stringsAsFactors = FALSE)
hubs       <- read.csv(paste0(data_path, "hubs.csv"),        stringsAsFactors = FALSE)

# ── Zone standardisation ─────────────────────────────────────────────────────
std_zone <- function(z) {
  z <- trimws(tolower(z))
  dplyr::case_when(
    z == "north"                      ~ "North",
    z == "south"                      ~ "South",
    z == "east"                       ~ "East",
    z == "west"                       ~ "West",
    z %in% c("central", "ctr")        ~ "Central",
    z == "airport"                    ~ "Airport",
    z %in% c("riverside", "riverSide")~ "Riverside",
    TRUE ~ tools::toTitleCase(z)
  )
}

orders$pickup_zone  <- std_zone(orders$pickup_zone)
orders$dropoff_zone <- std_zone(orders$dropoff_zone)
drivers$base_zone   <- std_zone(drivers$base_zone)
customers$home_zone <- std_zone(customers$home_zone)

# ── Type conversions ─────────────────────────────────────────────────────────
deliveries$customer_rating_post_delivery <- as.numeric(deliveries$customer_rating_post_delivery)
deliveries$fuel_or_charge_cost           <- as.numeric(deliveries$fuel_or_charge_cost)
deliveries$manual_route_override_count   <- as.integer(deliveries$manual_route_override_count)
deliveries$dispatch_time         <- ymd_hms(deliveries$dispatch_time)
deliveries$delivery_completed_at <- ymd_hms(deliveries$delivery_completed_at)
orders$order_created_at  <- ymd_hms(orders$order_created_at)
orders$order_value       <- as.numeric(orders$order_value)
drivers$driver_rating    <- as.numeric(drivers$driver_rating)
customers$loyalty_score  <- as.numeric(customers$loyalty_score)
complaints$resolution_days       <- as.numeric(complaints$resolution_days)
complaints$compensation_amount   <- as.numeric(complaints$compensation_amount)

# ── Feature engineering ──────────────────────────────────────────────────────
# Delivery duration in hours
deliveries <- deliveries %>%
  mutate(duration_hours = as.numeric(difftime(delivery_completed_at,
                                               dispatch_time, units = "hours")))

# Order month for trend analysis
orders <- orders %>%
  mutate(order_month = floor_date(order_created_at, "month"))

# Merge key tables
orders_deliveries <- orders %>%
  inner_join(deliveries, by = "order_id")

# Complaint count per customer
complaint_counts <- complaints %>%
  count(customer_id, name = "complaint_count")

customers <- customers %>%
  left_join(complaint_counts, by = "customer_id") %>%
  mutate(complaint_count = replace_na(complaint_count, 0))

cat("Data loaded, cleaned, and merged\n")
cat(sprintf("orders_deliveries: %d rows\n", nrow(orders_deliveries)))

---
## Analysis 1 — Descriptive Statistics

In [ ]:
cat("=== ORDER VALUE (n=", nrow(orders), ") ===\n")
print(summary(orders$order_value))
cat("SD:", round(sd(orders$order_value, na.rm = TRUE), 2), "\n\n")

cat("=== CUSTOMER RATING POST DELIVERY (n=",
    sum(!is.na(deliveries$customer_rating_post_delivery)), ") ===\n")
print(summary(deliveries$customer_rating_post_delivery))
cat("SD:", round(sd(deliveries$customer_rating_post_delivery, na.rm = TRUE), 2), "\n\n")

cat("=== COMPLAINT RESOLUTION DAYS (n=", nrow(complaints), ") ===\n")
print(summary(complaints$resolution_days))
cat("SD:", round(sd(complaints$resolution_days, na.rm = TRUE), 2), "\n\n")

cat("=== DRIVER RATING (n=", nrow(drivers), ") ===\n")
print(summary(drivers$driver_rating))
cat("SD:", round(sd(drivers$driver_rating, na.rm = TRUE), 2), "\n\n")

cat("=== DELIVERY STATUS COUNTS ===\n")
print(table(deliveries$delivery_status))
cat("\nDeliveries exceeding promised window:\n")
exceed <- orders_deliveries %>%
  filter(!is.na(duration_hours), duration_hours > 0,
         !is.na(promised_window_hours)) %>%
  summarise(
    total   = n(),
    exceeded = sum(duration_hours > promised_window_hours),
    pct     = round(100 * exceeded / total, 1)
  )
print(exceed)

### Interpretation — Descriptive Statistics
Order values range widely from £2.04 to £510.06 with a mean of £91.05 and a median of £76.58 — the positive skew indicates a small number of high-value orders pulling the mean upward. Customer ratings average 3.86 out of 5 with a standard deviation of 0.89, suggesting moderate but inconsistent satisfaction. Complaint resolution takes a mean of 7.93 days with a maximum of 25 days, indicating significant variation in case handling speed. Driver ratings are tightly clustered (mean 4.17, SD 0.41), which is notable because it suggests the rating system lacks discriminatory power — most drivers appear similarly rated despite wide variation in actual failure rates. Critically, 50.2% of deliveries with valid timestamps exceeded their promised window, meaning that half of all deliveries fail to meet the service commitment made to the customer.

---
## Analysis 2 — Correlation: Driver Rating vs Failure Rate

In [ ]:
# Calculate per-driver failure rate (minimum 3 deliveries)
driver_perf <- deliveries %>%
  group_by(driver_id) %>%
  summarise(
    total_deliveries = n(),
    failures         = sum(delivery_status == "Failed"),
    failure_rate     = failures / total_deliveries
  ) %>%
  filter(total_deliveries >= 3) %>%
  inner_join(drivers %>% select(driver_id, driver_rating, base_zone), by = "driver_id")

# Pearson correlation
corr_result <- cor.test(driver_perf$driver_rating, driver_perf$failure_rate,
                        method = "pearson")

cat("Pearson correlation — driver_rating vs failure_rate\n")
cat(sprintf("  r = %.4f\n", corr_result$estimate))
cat(sprintf("  p-value = %.4f\n", corr_result$p.value))
cat(sprintf("  95%% CI: [%.4f, %.4f]\n", corr_result$conf.int[1], corr_result$conf.int[2]))
cat(sprintf("  n = %d drivers\n", nrow(driver_perf)))

# Summary table: avg failure rate by rating quartile
driver_perf <- driver_perf %>%
  mutate(rating_quartile = ntile(driver_rating, 4))

quartile_summary <- driver_perf %>%
  group_by(rating_quartile) %>%
  summarise(
    avg_rating       = round(mean(driver_rating), 2),
    avg_failure_rate = round(mean(failure_rate) * 100, 1),
    n_drivers        = n()
  )
kable(quartile_summary, caption = "Table 5.1: Failure rate by driver rating quartile")

### Interpretation — Driver Rating vs Failure Rate
The Pearson correlation between driver rating and failure rate is r = -0.2021 (p ≈ 0.013), indicating a weak but statistically significant negative relationship. Higher-rated drivers tend to have slightly lower failure rates, but the relationship is too weak to rely on rating as a performance predictor. The quartile summary reinforces this: failure rates vary only modestly across rating bands, and some of the highest-rated drivers produce some of the highest failure rates, as seen in Query 5 of the SQL section. This suggests the driver rating system captures dimensions of performance (such as customer interaction quality) that do not correspond closely to operational delivery success.

---
## Analysis 3 — Correlation: Loyalty Score vs Complaint Frequency

In [ ]:
corr2 <- cor.test(customers$loyalty_score, customers$complaint_count,
                  method = "pearson", use = "complete.obs")

cat("Pearson correlation — loyalty_score vs complaint_count\n")
cat(sprintf("  r = %.4f\n", corr2$estimate))
cat(sprintf("  p-value = %.4f\n", corr2$p.value))
cat(sprintf("  n = %d customers\n", sum(!is.na(customers$loyalty_score))))

# Distribution of complaint counts
cat("\nComplaint count distribution across customers:\n")
print(table(customers$complaint_count))

### Interpretation — Loyalty Score vs Complaint Frequency
The correlation between loyalty score and complaint count is r = 0.026 (p ≈ 0.52), which is not statistically significant. This is a meaningful finding: it indicates that complaints are distributed broadly across the customer base regardless of loyalty tier, suggesting that service failures are not concentrated among dissatisfied or low-loyalty customers. High-loyalty customers are experiencing failures at similar rates to low-loyalty customers. This undermines any assumption that complaints are primarily driven by demanding or low-value customers, and supports the interpretation that the failures are systemic and operational in origin.

---
## Visualisation 1 — Delivery Status by Zone (Stacked Bar)

In [ ]:
zone_status <- orders_deliveries %>%
  count(pickup_zone, delivery_status) %>%
  group_by(pickup_zone) %>%
  mutate(pct = n / sum(n) * 100) %>%
  ungroup() %>%
  mutate(delivery_status = factor(delivery_status,
                                  levels = c("Failed", "Delayed", "OnTime")))

ggplot(zone_status, aes(x = reorder(pickup_zone, -pct, function(x) x[delivery_status == "Failed"]),
                        y = pct, fill = delivery_status)) +
  geom_col(width = 0.7) +
  geom_text(aes(label = ifelse(pct > 5, paste0(round(pct, 1), "%"), "")),
            position = position_stack(vjust = 0.5),
            size = 3.2, colour = "white", fontface = "bold") +
  scale_fill_manual(values = northstar_colours, name = "Delivery Status") +
  scale_y_continuous(labels = label_percent(scale = 1)) +
  labs(
    title    = "Figure 5.1: Delivery Status Breakdown by Zone",
    subtitle = "Ordered by failure rate (highest to lowest). Central and North are the worst-performing zones.",
    x        = "Pickup Zone",
    y        = "Percentage of Deliveries",
    caption  = "Source: NorthStar deliveries.csv + orders.csv"
  ) +
  theme_northstar

### Interpretation — Figure 5.1
The stacked bar chart makes the zone performance gap immediately visible. Central zone has the largest red (Failed) segment at 19.0%, while South has the smallest at 10.1%. The Airport zone's large amber (Delayed) segment — 27.4% — stands out even against zones with higher failure rates, confirming that Airport's problem is lateness rather than outright failure. The chart also shows that no zone achieves a failure rate below 10%, indicating that service reliability issues are network-wide rather than isolated to one area.

---
## Visualisation 2 — Customer Rating by Hub (Boxplot)

In [ ]:
hub_ratings <- deliveries %>%
  filter(!is.na(customer_rating_post_delivery)) %>%
  left_join(hubs %>% select(hub_id, hub_name, zone), by = "hub_id")

hub_means <- hub_ratings %>%
  group_by(hub_id, hub_name) %>%
  summarise(mean_rating = mean(customer_rating_post_delivery), .groups = "drop")

ggplot(hub_ratings,
       aes(x = reorder(hub_name, customer_rating_post_delivery, FUN = median),
           y = customer_rating_post_delivery)) +
  geom_boxplot(fill = "#1565C0", alpha = 0.3, outlier.colour = "#C62828",
               outlier.size = 1.5, width = 0.6) +
  geom_point(data = hub_means,
             aes(x = reorder(hub_name, mean_rating), y = mean_rating),
             shape = 18, size = 3, colour = "#C62828") +
  coord_flip() +
  labs(
    title    = "Figure 5.2: Customer Rating Distribution by Hub",
    subtitle = "Ordered by median rating. Red diamonds show mean. Red dots are outliers.",
    x        = "Hub",
    y        = "Customer Rating (1–5)",
    caption  = "Source: NorthStar deliveries.csv + hubs.csv"
  ) +
  theme_northstar

### Interpretation — Figure 5.2
Hub H05 records the lowest mean customer rating at 3.67 and the widest spread of scores, indicating that this hub produces the most variable and generally worst customer experience in the network. H02 and H04 perform best with means of 3.95 and 3.92 respectively. The presence of ratings as low as 1.0 at H01, H03, H05, and H06 highlights that very poor individual experiences are occurring across multiple hubs. The interquartile ranges are broadly similar across hubs, suggesting that the differences in mean rating are driven partly by the frequency of very low scores rather than a consistent shift in performance. Hub H05 should be a priority for operational investigation.

---
## Visualisation 3 — Order Value vs Fuel Cost by Service Type (Scatter)

In [ ]:
scatter_data <- orders_deliveries %>%
  filter(!is.na(order_value), !is.na(fuel_or_charge_cost))

ggplot(scatter_data,
       aes(x = fuel_or_charge_cost, y = order_value,
           colour = service_type, shape = delivery_status)) +
  geom_point(alpha = 0.45, size = 1.8) +
  geom_smooth(aes(group = 1), method = "lm", se = TRUE,
              colour = "#333333", linetype = "dashed", linewidth = 0.7) +
  scale_colour_brewer(palette = "Set2", name = "Service Type") +
  scale_shape_manual(values = c("OnTime" = 16, "Delayed" = 17, "Failed" = 4),
                     name = "Status") +
  labs(
    title    = "Figure 5.3: Order Value vs Fuel/Charge Cost by Service Type",
    subtitle = "Failed deliveries (x marks) incur cost without recovering full order value.",
    x        = "Fuel / Charge Cost (£)",
    y        = "Order Value (£)",
    caption  = "Source: NorthStar orders.csv + deliveries.csv"
  ) +
  theme_northstar

### Interpretation — Figure 5.3
The scatter plot shows that order value and fuel cost have a weak positive relationship overall — higher-value orders do not consistently cost more to deliver, which is expected given that fuel cost reflects distance rather than order value. Failed deliveries (x marks) are distributed throughout the cost range, confirming that failed orders still consume fuel and driver time. Parcel services show the widest value range, with some very high-value orders alongside low-cost deliveries. The dashed regression line indicates a modest positive trend but substantial variance, reinforcing that cost control requires operational rather than commercial intervention.

---
## Visualisation 4 — Monthly Order Volume and Failure Rate (Dual-Axis Line)

In [ ]:
monthly_trend <- orders %>%
  group_by(order_month) %>%
  summarise(order_count = n(), .groups = "drop") %>%
  left_join(
    orders_deliveries %>%
      mutate(order_month = floor_date(order_created_at, "month")) %>%
      group_by(order_month) %>%
      summarise(
        total_del  = n(),
        failed_del = sum(delivery_status == "Failed"),
        failure_rate = failed_del / total_del * 100
      ),
    by = "order_month"
  ) %>%
  filter(!is.na(order_month))

# Scale factor for dual axis
scale_factor <- max(monthly_trend$order_count, na.rm = TRUE) /
                max(monthly_trend$failure_rate, na.rm = TRUE)

ggplot(monthly_trend, aes(x = order_month)) +
  geom_col(aes(y = order_count), fill = "#1565C0", alpha = 0.35, width = 20) +
  geom_line(aes(y = failure_rate * scale_factor), colour = "#C62828",
            linewidth = 1.2) +
  geom_point(aes(y = failure_rate * scale_factor), colour = "#C62828", size = 2.2) +
  scale_y_continuous(
    name = "Order Volume",
    sec.axis = sec_axis(~ . / scale_factor, name = "Failure Rate (%)")
  ) +
  scale_x_datetime(date_labels = "%b %Y", date_breaks = "3 months") +
  labs(
    title    = "Figure 5.4: Monthly Order Volume and Delivery Failure Rate",
    subtitle = "Blue bars = order volume (left axis). Red line = failure rate % (right axis).",
    x        = NULL,
    caption  = "Source: NorthStar orders.csv + deliveries.csv"
  ) +
  theme_northstar +
  theme(axis.text.x = element_text(angle = 35, hjust = 1))

### Interpretation — Figure 5.4
Order volume fluctuates between approximately 37 and 66 orders per month across the dataset period (January 2024 to December 2025) with no clear upward trend. The failure rate, however, shows greater instability — spiking to 17.8% in May 2025 and 16.7% in July 2025, compared to a low of 1.8% in January 2024. Crucially, failure rate spikes do not consistently correspond to high-volume months, which suggests that volume pressure alone does not explain failures. The deteriorating failure rate in 2025 relative to 2024 is a concern: the average monthly failure rate in 2024 was approximately 9.5% versus approximately 11.1% in 2025, indicating a worsening trend that requires investigation.

---
## Visualisation 5 — Delivery Duration by Status (Violin Plot)

In [ ]:
duration_data <- deliveries %>%
  filter(!is.na(duration_hours),
         duration_hours > 0,
         duration_hours < 50) %>%   # Remove extreme outliers for clarity
  mutate(delivery_status = factor(delivery_status,
                                  levels = c("OnTime", "Delayed", "Failed")))

status_means <- duration_data %>%
  group_by(delivery_status) %>%
  summarise(mean_dur = mean(duration_hours), .groups = "drop")

ggplot(duration_data, aes(x = delivery_status, y = duration_hours,
                          fill = delivery_status)) +
  geom_violin(trim = FALSE, alpha = 0.5) +
  geom_boxplot(width = 0.12, fill = "white", outlier.shape = NA) +
  geom_point(data = status_means,
             aes(x = delivery_status, y = mean_dur),
             shape = 18, size = 4, colour = "#333333") +
  scale_fill_manual(values = northstar_colours, guide = "none") +
  labs(
    title    = "Figure 5.5: Delivery Duration by Outcome Status",
    subtitle = "Violin shows distribution shape. Inner box = IQR. Diamond = mean.",
    x        = "Delivery Status",
    y        = "Duration (hours)",
    caption  = "Source: NorthStar deliveries.csv. Observations > 50 hours excluded."
  ) +
  theme_northstar

### Interpretation — Figure 5.5
The violin plot reveals striking differences in duration distribution across delivery outcomes. OnTime deliveries have a median duration of 4.41 hours and a mean of 7.36 hours, with a distribution heavily concentrated at the lower end. Delayed deliveries show a broader, more symmetric distribution centred around 12.51 hours (median), extending to over 35 hours. Failed deliveries have a mean of 17.78 hours — more than double that of on-time deliveries — and show a wide, flat distribution indicating high variability in how long failed delivery attempts run before being recorded as failures. This suggests that failed deliveries are not being quickly identified and closed: drivers or systems are continuing to log time against orders that will ultimately not be completed, which has cost implications and distorts operational reporting.